<a href="https://colab.research.google.com/github/hyeonggyeong-kim/comprehensive-project/blob/khg/ml/model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# =========================
# 전체 데이터 병합 및 전처리
# =========================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from google.colab import files

# =========================
# STEP 1. 공통 컬럼 매핑 함수
# =========================

def rename_cols(df):
    return df.rename(columns={
        "시간":                          "TIMESTAMP",
        "차량 속도 센서":                 "SPEED",
        "엔진 회전수":                    "ENGINE_RPM",
        "액셀러레이터 페달 위치 D":        "THROTTLE_POS",
        "스로틀 위치 절대값":             "THROTTLE_ABS",
        "엔진 부하":                      "ENGINE_LOAD",
        "공기량 (MAF) 센서":              "MAF",
        "흡기 매니폴드 절대 압력 (MAP)":  "INTAKE_MANIFOLD_PRESSURE",
        "엔진 냉각 온도":                  "ENGINE_COOLANT_TEMP",
        "흡입 공기 온도 (IAT)":            "AIR_INTAKE_TEMP",
    })

def time_to_seconds(t):
    if isinstance(t, str):
        try:
            parts = t.strip().split(':')
            if len(parts) == 3:
                h, m, s = map(int, parts)
                return h * 3600 + m * 60 + s
        except:
            return np.nan
    return np.nan

# =========================
# STEP 2. 공개 데이터셋 (Kaggle)
# =========================

def clean_numeric(x):
    if isinstance(x, str):
        x = x.strip().replace('%','').replace(',','.')
        try:
            return float(x)
        except:
            return np.nan
    return x

df_public = pd.read_csv(
    "/content/sample_data/exp1_14drivers_14cars_dailyRoutes.csv",
    low_memory=False
)
df_public.columns = df_public.columns.str.strip()
for col in df_public.columns:
    df_public[col] = df_public[col].apply(clean_numeric)

df_public = df_public[[
    "TIMESTAMP","SPEED","ENGINE_RPM","THROTTLE_POS",
    "ENGINE_LOAD","ENGINE_COOLANT_TEMP","AIR_INTAKE_TEMP","TIMING_ADVANCE"
]].copy()
df_public["TIMESTAMP"] = pd.to_numeric(df_public["TIMESTAMP"], errors="coerce")
df_public["DATA_SOURCE"] = "public"
print(f"✅ 공개 데이터: {df_public.shape}")

# =========================
# STEP 3. 난폭운전 전용 데이터셋
# =========================

df_agr = pd.read_excel("/content/난폭운전_전용_대용량_데이터셋.xlsx")
df_agr = rename_cols(df_agr)
df_agr["TIMESTAMP"] = df_agr["TIMESTAMP"].apply(time_to_seconds)
df_agr["TIMING_ADVANCE"] = 0
df_agr["DATA_SOURCE"] = "aggressive"
print(f"✅ 난폭운전 데이터: {df_agr.shape}")

# =========================
# STEP 4. 통합 주행정보 (실차)
# =========================

df_unified = pd.read_excel("/content/통합_주행정보.xlsx")
df_unified = rename_cols(df_unified)
df_unified["TIMESTAMP"] = df_unified["TIMESTAMP"].apply(time_to_seconds)
df_unified["TIMING_ADVANCE"] = 0
df_unified["DATA_SOURCE"] = "real_unified"
print(f"✅ 통합 주행 데이터: {df_unified.shape}")

# =========================
# STEP 5. 개별 주행 데이터 1~4
# =========================

real_dfs = []
for i in range(1, 5):
    df_tmp = pd.read_excel(
        f"/content/주행 정보 {i}.xls",
        header=1  # ← 2번째 행이 실제 헤더
    )
    df_tmp = rename_cols(df_tmp)
    df_tmp["TIMESTAMP"] = df_tmp["TIMESTAMP"].apply(time_to_seconds)
    df_tmp["TIMING_ADVANCE"] = 0
    df_tmp["DATA_SOURCE"] = f"real_{i}"
    real_dfs.append(df_tmp)
    print(f"✅ 주행 {i} 데이터: {df_tmp.shape}")

df_real_all = pd.concat(real_dfs, ignore_index=True)
print(f"✅ 개별 주행 통합: {df_real_all.shape}")

# =========================
# STEP 6. 공통 컬럼으로 통일 후 병합
# =========================

common_cols = [
    "TIMESTAMP", "SPEED", "ENGINE_RPM", "THROTTLE_POS",
    "ENGINE_LOAD", "ENGINE_COOLANT_TEMP", "AIR_INTAKE_TEMP",
    "TIMING_ADVANCE", "DATA_SOURCE"
]

def select_cols(df):
    for col in common_cols:
        if col not in df.columns:
            df[col] = 0
    return df[common_cols]

df_all = pd.concat([
    select_cols(df_public),
    select_cols(df_agr),
    select_cols(df_unified),
    select_cols(df_real_all),
], ignore_index=True)

print(f"\n병합 결과:")
print(f"  공개 데이터:    {len(df_public)}건")
print(f"  난폭운전:       {len(df_agr)}건")
print(f"  통합 주행:      {len(df_unified)}건")
print(f"  개별 주행 1~4:  {len(df_real_all)}건")
print(f"  전체 합계:      {len(df_all)}건")
print(f"  DATA_SOURCE 분포:")
print(df_all["DATA_SOURCE"].value_counts())

# =========================
# STEP 7. 숫자형 변환 및 이상치 처리
# =========================

for col in common_cols:
    if col not in ["TIMESTAMP", "DATA_SOURCE"]:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

if "SPEED" in df_all.columns:
    df_all.loc[df_all["SPEED"] < 0, "SPEED"] = np.nan
if "ENGINE_RPM" in df_all.columns:
    df_all.loc[df_all["ENGINE_RPM"] < 0, "ENGINE_RPM"] = np.nan
if "THROTTLE_POS" in df_all.columns:
    df_all.loc[
        (df_all["THROTTLE_POS"] < 0) | (df_all["THROTTLE_POS"] > 100),
        "THROTTLE_POS"
    ] = np.nan

numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy="median")
df_all[numeric_cols] = imputer.fit_transform(df_all[numeric_cols])

print(f"\n결측치 처리 후: {df_all.isnull().sum().sum()}개")

# =========================
# STEP 8. 파생변수 생성
# =========================

window_size = 5

df_all["SPEED_DIFF"]    = df_all["SPEED"].diff().fillna(0)
df_all["RPM_DIFF"]      = df_all["ENGINE_RPM"].diff().fillna(0)
df_all["THROTTLE_DIFF"] = df_all["THROTTLE_POS"].diff().fillna(0)

df_all["SPEED_MA"]      = df_all["SPEED"].rolling(window=window_size, min_periods=1).mean()
df_all["SPEED_STD"]     = df_all["SPEED"].rolling(window=window_size, min_periods=1).std().fillna(0)
df_all["RPM_MA"]        = df_all["ENGINE_RPM"].rolling(window=window_size, min_periods=1).mean()
df_all["RPM_STD"]       = df_all["ENGINE_RPM"].rolling(window=window_size, min_periods=1).std().fillna(0)
df_all["THROTTLE_MA"]   = df_all["THROTTLE_POS"].rolling(window=window_size, min_periods=1).mean()
df_all["THROTTLE_STD"]  = df_all["THROTTLE_POS"].rolling(window=window_size, min_periods=1).std().fillna(0)

# =========================
# STEP 9. 운전 점수 산출
# =========================

def calculate_driving_score(row):
    score = 100
    if "SPEED_DIFF" in row:
        accel = max(0, row["SPEED_DIFF"] - 5)
        score -= min(30, accel * 1.5)
        brake = max(0, -row["SPEED_DIFF"] - 5)
        score -= min(30, brake * 1.5)
    if "ENGINE_RPM" in row and row["ENGINE_RPM"] > 2500:
        rpm_excess = (row["ENGINE_RPM"] - 2500) / 100
        score -= min(20, rpm_excess * 2)
    if "SPEED" in row and row["SPEED"] > 80:
        speed_excess = row["SPEED"] - 80
        score -= min(25, speed_excess * 0.5)
    if "SPEED_STD" in row and row["SPEED_STD"] > 5:
        score -= min(15, (row["SPEED_STD"] - 5) * 1.0)
    return max(0, round(score, 1))

df_all["DRIVING_SCORE"] = df_all.apply(calculate_driving_score, axis=1)

print("\n===== 최종 운전 점수 분포 =====")
print(df_all["DRIVING_SCORE"].describe())
print("\n점수 구간별 분포:")
print(pd.cut(df_all["DRIVING_SCORE"],
             bins=[-1, 40, 80, 100],
             labels=["aggressive(0~40)", "normal(41~80)", "safe(81~100)"]
             ).value_counts())

# 출처별 점수 분포
print("\n출처별 평균 점수:")
print(df_all.groupby("DATA_SOURCE")["DRIVING_SCORE"].mean().round(2))

# =========================
# STEP 10. X, y 분리 및 학습
# =========================

target_col = "DRIVING_SCORE"
drop_cols  = ["TIMESTAMP", "DATA_SOURCE", target_col, "SPEED_STD"]
drop_cols  = [c for c in drop_cols if c in df_all.columns]

X_all = df_all.drop(columns=drop_cols)
y_all = df_all[target_col]

print(f"\n사용 피처 ({X_all.shape[1]}개): {list(X_all.columns)}")
print(f"X shape: {X_all.shape}")
print(f"y shape: {y_all.shape}")

# train/test split
from sklearn.model_selection import train_test_split
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# 스케일링
from sklearn.preprocessing import StandardScaler
scaler_all = StandardScaler()
X_train_all_sc = scaler_all.fit_transform(X_train_all)
X_test_all_sc  = scaler_all.transform(X_test_all)

print(f"\nX_train: {X_train_all.shape}")
print(f"X_test : {X_test_all.shape}")

# =========================
# STEP 11. 다양한 모델 비교 (요청된 15개 모델 반영)
# =========================

import time
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              AdaBoostRegressor, GradientBoostingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Transformer 모델은 scikit-learn에 내장되어 있지 않아,
# 원활한 코드 실행을 위해 유사한 딥러닝 기반인 MLPRegressor를 대체재로 사용했습니다.
models_all = {
    "Ridge":         Ridge(random_state=42),
    "Lasso":         Lasso(random_state=42),
    "ElasticNet":    ElasticNet(random_state=42),
    "DecisionTree":  DecisionTreeRegressor(random_state=42),
    "RandomForest":  RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "ExtraTrees":    ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "AdaBoost":      AdaBoostRegressor(random_state=42),
    "GradBoost":     GradientBoostingRegressor(random_state=42),
    "SVR_RBF":       SVR(kernel='rbf'),
    "SVR_Poly":      SVR(kernel='poly'),
    "SVR_Linear":    SVR(kernel='linear'),
    "KNN":           KNeighborsRegressor(n_jobs=-1),
    "LightGBM":      LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost":       XGBRegressor(random_state=42, n_jobs=-1),
    "Transformer":   MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42) # 딥러닝 대체재
}

all_results = []
trained_models_all = {}

print("\n===== 전체 병합 데이터 모델 성능 =====")
print(f"{'Model':<15} {'MAE':>8} {'MSE':>10} {'RMSE':>8} {'R²':>8} {'Speed':>8}")
print("-" * 70)

# 값의 스케일(크기)에 민감한 모델들은 스케일링된 데이터를 사용해야 합니다.
scaled_models = ["Ridge", "Lasso", "ElasticNet", "SVR_RBF", "SVR_Poly", "SVR_Linear", "KNN", "Transformer"]

for model_name, model in models_all.items():
    # 모델 특성에 맞춰 스케일링된 데이터(sc)와 원본 데이터 분기 처리
    if model_name in scaled_models:
        X_tr = X_train_all_sc
        X_te = X_test_all_sc
    else:
        X_tr = X_train_all
        X_te = X_test_all

    start = time.time()
    try:
        model.fit(X_tr, y_train_all)
        elapsed = round(time.time() - start, 4)

        y_pred = model.predict(X_te)
        mae  = mean_absolute_error(y_test_all, y_pred)
        mse  = mean_squared_error(y_test_all, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test_all, y_pred)

        all_results.append({
            "Model": model_name,
            "MAE":   round(mae,  4),
            "MSE":   round(mse,  4),
            "RMSE":  round(rmse, 4),
            "R2":    round(r2,   4),
            "Speed": elapsed
        })
        trained_models_all[model_name] = model

        print(f"{model_name:<15} {mae:>8.4f} {mse:>10.4f} {rmse:>8.4f} {r2:>8.4f} {elapsed:>8.4f}s")

    except Exception as e:
        print(f"{model_name:<15} Failed: {e}")

all_results_df = pd.DataFrame(all_results).sort_values("MAE")

# =========================
# STEP 12. 결과 저장
# =========================

df_all.to_csv("/content/obd_all_merged.csv", index=False)
all_results_df.to_csv("/content/all_merged_model_results.csv", index=False)
print("\n✅ 저장 완료: obd_all_merged.csv")
print(f"전체 크기: {df_all.shape}")

✅ 공개 데이터: (60439, 9)
✅ 난폭운전 데이터: (6000, 18)
✅ 통합 주행 데이터: (25008, 30)
✅ 주행 1 데이터: (1324, 30)
✅ 주행 2 데이터: (1060, 30)
✅ 주행 3 데이터: (827, 30)
✅ 주행 4 데이터: (944, 30)
✅ 개별 주행 통합: (4155, 30)

병합 결과:
  공개 데이터:    60439건
  난폭운전:       6000건
  통합 주행:      25008건
  개별 주행 1~4:  4155건
  전체 합계:      95602건
  DATA_SOURCE 분포:
DATA_SOURCE
public          60439
real_unified    25008
aggressive       6000
real_1           1324
real_2           1060
real_4            944
real_3            827
Name: count, dtype: int64

결측치 처리 후: 0개

===== 최종 운전 점수 분포 =====
count    95602.000000
mean        94.261152
std         10.827998
min         12.800000
25%         93.300000
50%        100.000000
75%        100.000000
max        100.000000
Name: DRIVING_SCORE, dtype: float64

점수 구간별 분포:
DRIVING_SCORE
safe(81~100)        84752
normal(41~80)       10809
aggressive(0~40)       41
Name: count, dtype: int64

출처별 평균 점수:
DATA_SOURCE
aggressive      87.83
public          92.60
real_1          98.51
real_2          99.19
real_